# One adder, four libraries
A companion to *Logic Synthesis from the Ground Up*, article 2, by Dheepak Jayaraman.

The same RTL, a half adder and an 8-bit adder, is mapped with Yosys and ABC against four versions of the open Nangate 45 nm library. A small table-based calculator then times each result. Run the cells from top to bottom. Nothing to install on your own machine.

**How to read the numbers.** Area is in library units. Delay assumes a 20 ps input slew, 3 fF on every output and no wires. The calculator is not sign-off timing, so compare rows with each other and do not quote the absolute delays. One mapper, one run per row. Commercial tools will choose differently.

## 1. Install Yosys

In [ ]:
!pip install -q yowasp-yosys
!yowasp-yosys -V

## 2. Write the files
The designs, the mapper constraints, the library cutter, the delay calculator and the driver script. You can open and edit any of them from the file panel on the left.

In [ ]:
%%writefile add.v
// Two designs: a 1-bit half adder and an 8-bit adder with two 8-bit inputs
module ha(input a, b, output s, c);
  assign s = a ^ b;
  assign c = a & b;
endmodule

module add8(input [7:0] a, b, output [8:0] s);
  assign s = a + b;
endmodule

In [ ]:
%%writefile c.constr
set_driving_cell BUF_X1
set_load 3.0

In [ ]:
%%writefile cut_lib.py
"""cut_lib.py: write four reduced copies of the Nangate45 Liberty file.
usage: python3 cut_lib.py NangateOpenCellLibrary_typical.lib
Writes lib_full.lib, lib_noxor.lib, lib_nand.lib, lib_x1.lib in the current directory."""
import re, sys
src = open(sys.argv[1]).read()

def cell_blocks(s):
    out = []
    for m in re.finditer(r'\n\s*cell\s*\(\s*"?([A-Za-z0-9_]+)"?\s*\)\s*\{', s):
        start = m.start() + 1
        depth, j = 0, m.end() - 1
        while True:
            if s[j] == '{': depth += 1
            elif s[j] == '}':
                depth -= 1
                if depth == 0: break
            j += 1
        out.append((m.group(1), start, j + 1))
    return out

blocks = cell_blocks(src)
# cells the tools cannot use for plain combinational mapping, or that this study does not use
skip = re.compile(r'^(FILL|TAP|ANTENNA|LOGIC|CLKGATE|DFF|DLH|DLL|SDFF|TBUF|TINV|HA_|FA_)')

def make(name, keep):
    s = src
    for n, a, b in sorted(blocks, key=lambda x: -x[1]):
        if not keep(n):
            s = s[:a] + s[b:]
    open(name, 'w').write(s)
    print(name, sum(1 for n, _, _ in blocks if keep(n)), 'cells')

usable = lambda n: not skip.match(n)
make('lib_full.lib',  lambda n: usable(n))
make('lib_noxor.lib', lambda n: usable(n) and not n.startswith(('XOR', 'XNOR')))
make('lib_nand.lib',  lambda n: n in ('NAND2_X1', 'NOR2_X1', 'INV_X1', 'BUF_X1'))
make('lib_x1.lib',    lambda n: usable(n) and n.endswith('_X1'))

In [ ]:
%%writefile sta.py
"""sta.py: tiny NLDM static timing for a mapped gate-level Verilog netlist (worst-case, no wires).
usage: python3 sta.py lib.lib netlist.v [input_slew_ns] [out_load_fF]"""
import re,sys,bisect
def tokenize(s):
    return re.findall(r'"[^"]*"|[A-Za-z_][\w\.\[\]\$]*|[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?|[{}();:,\\]',s)
def parse(path):
    s=open(path).read()
    s=re.sub(r'/\*.*?\*/','',s,flags=re.S); s=re.sub(r'//[^\n]*','',s); s=s.replace('\\\n',' ')
    t=tokenize(s); i=0
    def group():
        nonlocal i
        # at: name ( args ) {  OR  name : value ;  OR name ( args ) ;
        name=t[i]; i+=1
        if t[i]==':':
            i+=1; v=[]
            while t[i]!=';': v.append(t[i]); i+=1
            i+=1; return ('attr',name,' '.join(v))
        assert t[i]=='(',(name,t[i])
        i+=1; args=[]
        while t[i]!=')':
            if t[i]!=',': args.append(t[i].strip('"'))
            i+=1
        i+=1
        if t[i]=='{':
            i+=1; kids=[]
            while t[i]!='}': kids.append(group())
            i+=1; return ('group',name,args,kids)
        if t[i]==';': i+=1
        return ('cattr',name,args)
    top=group(); return top
def floats(a): return [float(x) for x in re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?',' '.join(a))]
def load(path):
    top=parse(path); tmpl={}; cells={}
    for k in top[3]:
        if k[0]=='group' and k[1]=='lu_table_template':
            d={}
            for a in k[3]:
                if a[0]=='cattr': d[a[1]]=floats(a[2])
            tmpl[k[2][0]]=d
    for k in top[3]:
        if not(k[0]=='group' and k[1]=='cell'): continue
        c=dict(area=0,pins={},arcs=[]); 
        for a in k[3]:
            if a[0]=='attr' and a[1]=='area': c['area']=float(a[2])
            if a[0]=='group' and a[1]=='pin':
                pn=a[2][0]; p=dict(dir=None,cap=0.0)
                for b in a[3]:
                    if b[0]=='attr' and b[1]=='direction': p['dir']=b[2]
                    if b[0]=='attr' and b[1]=='capacitance': p['cap']=float(b[2])
                c['pins'][pn]=p
                for b in a[3]:
                    if b[0]=='group' and b[1]=='timing':
                        arc=dict(out=pn,rel=None,sense='non_unate',tabs={})
                        for e in b[3]:
                            if e[0]=='attr' and e[1]=='related_pin': arc['rel']=e[2].strip('"')
                            if e[0]=='attr' and e[1]=='timing_sense': arc['sense']=e[2]
                            if e[0]=='group' and e[1] in('cell_rise','cell_fall','rise_transition','fall_transition'):
                                d=dict(i1=None,i2=None,v=None)
                                for f in e[3]:
                                    if f[0]=='cattr':
                                        if f[1]=='index_1': d['i1']=floats(f[2])
                                        if f[1]=='index_2': d['i2']=floats(f[2])
                                        if f[1]=='values': d['v']=floats(f[2])
                                tm=tmpl.get(e[2][0],{})
                                if d['i1'] is None: d['i1']=tm.get('index_1')
                                if d['i2'] is None: d['i2']=tm.get('index_2')
                                if d['i2'] is None or d['i1'] is None: continue
                                n2=len(d['i2']); d['v']=[d['v'][r*n2:(r+1)*n2] for r in range(len(d['i1']))]
                                arc['tabs'][e[1]]=d
                        if arc['rel'] and 'cell_rise' in arc['tabs']: c['arcs'].append(arc)
        cells[k[2][0]]=c
    return cells
def interp(tab,x,y):
    i1,i2,v=tab['i1'],tab['i2'],tab['v']
    def seg(a,x):
        j=bisect.bisect_right(a,x)-1; j=max(0,min(j,len(a)-2)); return j,(x-a[j])/(a[j+1]-a[j])
    j,fx=seg(i1,x); k,fy=seg(i2,y)
    a=v[j][k]*(1-fy)+v[j][k+1]*fy; b=v[j+1][k]*(1-fy)+v[j+1][k+1]*fy
    return a*(1-fx)+b*fx
def netlist(path):
    s=open(path).read(); s=re.sub(r'/\*.*?\*/','',s,flags=re.S); s=re.sub(r'//[^\n]*','',s)
    ports={'input':[],'output':[]}
    for m in re.finditer(r'\b(input|output)\s*(\[(\d+):(\d+)\])?\s*([\w\\\$\.]+)\s*;',s):
        d,_,hi,lo,n=m.groups()
        if hi is None: ports[d].append(n)
        else: ports[d]+= [f'{n}[{b}]' for b in range(int(lo),int(hi)+1)]
    insts=[]
    for m in re.finditer(r'^\s*([A-Z][A-Z0-9_]*_X\d+)\s+(\S+)\s*\((.*?)\);',s,re.S|re.M):
        pins=dict((p,n.strip()) for p,n in re.findall(r'\.(\w+)\(([^()]*)\)',m.group(3)))
        insts.append((m.group(1),m.group(2),pins))
    assigns=re.findall(r'assign\s+([^=;]+?)\s*=\s*([^;]+?)\s*;',s)
    return ports,insts,assigns
def run(lib,nl,slew=0.02,outload=3.0):
    cells=load(lib); ports,insts,assigns=netlist(nl)
    alias={}
    for l,r in assigns: alias[l.strip()]=r.strip()
    def res(n):
        n=n.strip()
        while n in alias: n=alias[n]
        return n
    load_={}; drv={}
    for ct,nm,pins in insts:
        c=cells[ct]
        for p,n in pins.items():
            n=res(n)
            if c['pins'][p]['dir']=='input': load_[n]=load_.get(n,0)+c['pins'][p]['cap']
            else: drv[n]=(ct,nm,pins)
    for o in ports['output']: load_[res(o)]=load_.get(res(o),0)+outload
    arr={}  # net -> {'r':(t,slew),'f':(t,slew)}
    for n in ports['input']: arr[n]={'r':(0.0,slew),'f':(0.0,slew)}
    for n in('1\'h0','1\'h1',"1'b0","1'b1"): arr[n]={'r':(0.0,0.0),'f':(0.0,0.0)}
    import sys; sys.setrecursionlimit(10000)
    def get(n):
        n=res(n)
        if n in arr: return arr[n]
        ct,nm,pins=drv[n]; c=cells[ct]
        outp=[p for p,x in pins.items() if res(x)==n][0]
        best={'r':(0.0,0.0),'f':(0.0,0.0)}; cl=load_.get(n,0.0)
        for a in c['arcs']:
            if a['out']!=outp or a['rel'] not in pins: continue
            ia=get(pins[a['rel']])
            for oe,(tk,sk) in (('r',('cell_rise','rise_transition')),('f',('cell_fall','fall_transition'))):
                if tk not in a['tabs']: continue
                ies={'positive_unate':[oe],'negative_unate':['f' if oe=='r' else 'r']}.get(a['sense'],['r','f'])
                for ie in ies:
                    t0,s0=ia[ie]; d=interp(a['tabs'][tk],s0,cl); so=interp(a['tabs'][sk],s0,cl)
                    if t0+d>best[oe][0]: best[oe]=(t0+d,so)
        arr[n]=best; return best
    wd=0.0; wn=None
    for o in ports['output']:
        a=get(o)
        for e in 'rf':
            if a[e][0]>wd: wd=a[e][0]; wn=o
    area=sum(cells[ct]['area'] for ct,_,_ in insts)
    return wd*1000,area,len(insts),wn
if __name__=='__main__':
    lib,nl=sys.argv[1],sys.argv[2]; sl=float(sys.argv[3]) if len(sys.argv)>3 else 0.02; ol=float(sys.argv[4]) if len(sys.argv)>4 else 3.0
    d,a,n,o=run(lib,nl,sl,ol); print(f'delay={d:.1f} ps area={a:.3f} cells={n} worst_out={o}')

In [ ]:
%%writefile run_all.py
"""run_all.py: map each design against each library variant and time the result.
usage: python3 run_all.py
Needs yosys (or yowasp-yosys) on the PATH, the four lib_*.lib files from cut_lib.py,
add.v, c.constr and sta.py in the current directory."""
import os, re, shutil, subprocess, sys
from collections import Counter
import sta

YOSYS = shutil.which('yosys') or shutil.which('yowasp-yosys')
if not YOSYS: sys.exit('yosys not found on PATH')
LIBS = [('Full library', 'lib_full'), ('No XOR or XNOR cells', 'lib_noxor'),
        ('NAND2, NOR2, INV and BUF only', 'lib_nand'), ('Full library, X1 drive only', 'lib_x1')]
TARGETS = [('tight', 100), ('relaxed', 600)]      # ABC delay target in ps
IN_SLEW, OUT_LOAD = 0.02, 3.0                      # ns, fF

def map_design(top, lib, target):
    out = f'nl_{top}_{lib}_{target}.v'
    if os.path.exists(out): os.remove(out)
    cmd = (f'read_verilog add.v; synth -noabc -top {top} -flatten; '
           f'abc -liberty {lib}.lib -D {target} -constr c.constr; '
           f'opt_clean -purge; write_verilog -noattr {out}')
    # run from this directory with relative names: the sandboxed yowasp build only sees the cwd
    subprocess.run([YOSYS, '-p', cmd], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return out if os.path.exists(out) else None

for top in ('ha', 'add8'):
    print(f'\n== {top} ==')
    print(f'{"library":32s} {"target":8s} {"cells":>5s} {"area":>7s} {"delay ps":>9s}  cells used')
    for label, lib in LIBS:
        for tname, tps in TARGETS:
            nl = map_design(top, lib, tps)
            if not nl:
                print(f'{label:32s} {tname:8s} mapping failed'); continue
            delay, area, ncells, _ = sta.run(lib + '.lib', nl, IN_SLEW, OUT_LOAD)
            used = Counter(re.findall(r'^\s*([A-Z][A-Z0-9]*_X\d+)\s', open(nl).read(), re.M))
            print(f'{label:32s} {tname:8s} {ncells:5d} {area:7.2f} {delay:9.1f}  ' +
                  ' '.join(f'{k}:{v}' for k, v in sorted(used.items())))

# Extra check: the unmodified library also holds half-adder and full-adder cells. Are they used?
orig = 'NangateOpenCellLibrary_typical'
if os.path.exists(orig + '.lib'):
    print('\n== unmodified Nangate library (includes HA_X1 and FA_X1) ==')
    for top in ('ha', 'add8'):
        nl = map_design(top, orig, 100)
        if not nl: print(top, 'mapping failed'); continue
        text = open(nl).read()
        print(f'{top}: HA_X1 used {len(re.findall(r"^\s*HA_X1", text, re.M))} times, '
              f'FA_X1 used {len(re.findall(r"^\s*FA_X1", text, re.M))} times')

## 3. Download the library and build the four variants

In [ ]:
!curl -sSL -o NangateOpenCellLibrary_typical.lib https://raw.githubusercontent.com/The-OpenROAD-Project/OpenROAD-flow-scripts/master/flow/platforms/nangate45/lib/NangateOpenCellLibrary_typical.lib
!python3 cut_lib.py NangateOpenCellLibrary_typical.lib

## 4. Run the experiment
Tight asks the mapper for the fastest netlist it can find (100 ps target). Relaxed gives it 600 ps, which it does not need. It only takes a short while.

In [ ]:
!python3 run_all.py

## 5. Things to try
- In `add.v`, change `add8` to 16 bits and rerun.
- In `run_all.py`, add a fifth library, for example one with only AND2, OR2 and INV, by adding a line to `cut_lib.py`.
- Change `TARGETS` in `run_all.py` to see where the tool starts upsizing cells.
- Look at a mapped netlist, for example `!cat nl_ha_lib_noxor_100.v`.

**Results from my run, for comparison.** Full library, tight: 257 ps. No XOR or XNOR cells, tight: 371 ps. The Nangate library contains half-adder and full-adder cells, and the last lines of the output show that the tool never uses them. Small differences in your Yosys version can move individual numbers a little.